# 01 Data Preparation

This notebook prepares the daily PAGASA Science Garden rainfall dataset for rainfall trend analysis.

Main tasks:

1. Load daily Science Garden observations.
2. Clean rainfall values, including missing and trace rainfall codes.
3. Load IBTrACS western North Pacific tropical cyclone tracks.
4. Calculate tropical cyclone distance to Science Garden.
5. Create daily tropical cyclone proximity flags.
6. Merge rainfall observations with TC proximity information.
7. Export a local prepared dataset for analysis.

Raw PAGASA data and local intermediate files are not intended for public redistribution.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

PROJECT_DIR = Path(".").resolve()

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
TABLE_DIR = OUTPUT_DIR / "tables"
INTERMEDIATE_DIR = OUTPUT_DIR / "intermediate"

for path in [OUTPUT_DIR, TABLE_DIR, INTERMEDIATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SCI_GARDEN_PATH = DATA_DIR / "sci-garden-daily.csv"
IBTRACS_PATH = DATA_DIR / "ibtracs.WP.list.v04r01.csv"

PREPARED_CSV_PATH = INTERMEDIATE_DIR / "science_garden_daily_prepared.csv"
PREPARED_PKL_PATH = INTERMEDIATE_DIR / "science_garden_daily_prepared.pkl"
PREPARED_METADATA_PATH = INTERMEDIATE_DIR / "science_garden_daily_prepared_metadata.json"

ANNUAL_COMPLETENESS_PATH = TABLE_DIR / "annual_data_completeness.csv"
MONTHLY_COMPLETENESS_PATH = TABLE_DIR / "monthly_data_completeness.csv"

In [2]:
# Station location
SCI_GARDEN_LAT = 14.65
SCI_GARDEN_LON = 121.05

# Rainfall definitions
MISSING_RAINFALL_CODE = -999.0
TRACE_RAINFALL_CODE = -1.0
TRACE_REPLACEMENT_VALUE = 0.0
WET_DAY_THRESHOLD = 1.0

# TC proximity settings
TC_RADII_KM = [250, 500, 1000]
PRIMARY_TC_RADIUS_KM = 1000

# Completeness threshold used later in analysis
MIN_VALID_FRACTION = 0.90

In [3]:
required_files = [SCI_GARDEN_PATH, IBTRACS_PATH]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Required input file not found: {file_path}")

print("Input files found:")
for file_path in required_files:
    print(f"- {file_path}")

Input files found:
- /home/jupyter-bbr/source/science-garden-rainfall-trends/data/sci-garden-daily.csv
- /home/jupyter-bbr/source/science-garden-rainfall-trends/data/ibtracs.WP.list.v04r01.csv


In [4]:
def normalize_lon_180(lon):
    """
    Normalize longitude to the -180 to 180 range.
    Works with scalars, NumPy arrays, and pandas Series.
    """
    return ((lon + 180) % 360) - 180


def haversine_km(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance in kilometers.
    
    lat1, lon1:
        Scalar station latitude and longitude.
    
    lat2, lon2:
        Arrays or pandas Series of tropical cyclone positions.
    """
    radius_earth_km = 6371.0
    
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(normalize_lon_180(lon1))
    
    lat2_rad = np.radians(pd.Series(lat2, dtype="float64"))
    lon2_rad = np.radians(normalize_lon_180(pd.Series(lon2, dtype="float64")))
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    )
    
    c = 2 * np.arcsin(np.sqrt(a))
    
    return radius_earth_km * c


def expected_days_in_year(year):
    """
    Return expected number of days in a calendar year.
    """
    return pd.Timestamp(year=int(year), month=12, day=31).dayofyear


def expected_days_in_month(year, month):
    """
    Return expected number of days in a calendar month.
    """
    return pd.Timestamp(year=int(year), month=int(month), day=1).days_in_month

## Load and clean Science Garden rainfall

The Science Garden daily file contains daily rainfall and other meteorological variables. Rainfall cleaning follows these conventions:

- `-999` is treated as missing.
- `-1.0` represents trace rainfall and is retained as a valid observation but converted to `0.0 mm`.
- A wet day is defined as rainfall greater than or equal to `1.0 mm`.

In [5]:
sci = pd.read_csv(
    SCI_GARDEN_PATH,
    sep=None,
    engine="python"
)

sci.columns = sci.columns.str.strip().str.upper()

required_columns = ["YEAR", "MONTH", "DAY", "RAINFALL"]
missing_columns = [col for col in required_columns if col not in sci.columns]

if missing_columns:
    raise ValueError(f"Missing required Science Garden columns: {missing_columns}")

sci["DATE"] = pd.to_datetime(
    sci[["YEAR", "MONTH", "DAY"]],
    errors="coerce"
)

sci = sci.dropna(subset=["DATE"]).copy()
sci["DATE"] = sci["DATE"].dt.floor("D")

print("Loaded Science Garden daily data")
print(f"Rows: {len(sci):,}")
print(f"Date range: {sci['DATE'].min().date()} to {sci['DATE'].max().date()}")

display(sci.head())

Loaded Science Garden daily data
Rows: 23,376
Date range: 1961-01-01 to 2024-12-31


,YEAR,MONTH,DAY,RAINFALL,TMAX,TMIN,RH,WIND_SPEED,WIND_DIRECTION,DATE
0,1961,1,1,-999.0,-999.0,-999.0,-999,-999,-999,1961-01-01
1,1961,1,2,-999.0,-999.0,-999.0,-999,-999,-999,1961-01-02
2,1961,1,3,-999.0,-999.0,-999.0,-999,-999,-999,1961-01-03
3,1961,1,4,-999.0,-999.0,-999.0,-999,-999,-999,1961-01-04
4,1961,1,5,0.0,29.7,18.2,75,2,90,1961-01-05


In [6]:
sci = sci.copy()

# Convert known numeric columns if present
numeric_columns = [
    "YEAR",
    "MONTH",
    "DAY",
    "RAINFALL",
    "TMAX",
    "TMIN",
    "RH",
    "WIND_SPEED",
    "WIND_DIRECTION",
]

for col in numeric_columns:
    if col in sci.columns:
        sci[col] = pd.to_numeric(sci[col], errors="coerce")

# Preserve original rainfall coding before cleaning
sci["RAINFALL_RAW"] = sci["RAINFALL"]

# Rainfall flags
sci["RAINFALL_MISSING_CODE"] = sci["RAINFALL_RAW"] == MISSING_RAINFALL_CODE
sci["RAINFALL_TRACE"] = sci["RAINFALL_RAW"] == TRACE_RAINFALL_CODE

# Clean rainfall
sci["RAINFALL"] = sci["RAINFALL_RAW"].replace({
    MISSING_RAINFALL_CODE: np.nan,
    TRACE_RAINFALL_CODE: TRACE_REPLACEMENT_VALUE,
})

# Replace -999 in other numeric meteorological variables with NaN
for col in numeric_columns:
    if col in sci.columns and col not in ["YEAR", "MONTH", "DAY", "RAINFALL"]:
        sci[col] = sci[col].replace(MISSING_RAINFALL_CODE, np.nan)

# Check for unexpected negative rainfall values
unexpected_negative_rain = sci[
    sci["RAINFALL"].notna() & (sci["RAINFALL"] < 0)
].copy()

print("Rainfall cleaning summary")
print("-------------------------")
print(f"Trace rainfall days: {sci['RAINFALL_TRACE'].sum():,}")
print(f"Missing rainfall code days: {sci['RAINFALL_MISSING_CODE'].sum():,}")
print(f"Missing rainfall after cleaning: {sci['RAINFALL'].isna().sum():,}")
print(f"Unexpected negative rainfall values after cleaning: {len(unexpected_negative_rain):,}")

if len(unexpected_negative_rain) > 0:
    display(unexpected_negative_rain[["DATE", "RAINFALL_RAW", "RAINFALL"]].head(20))

display(
    sci[["DATE", "RAINFALL_RAW", "RAINFALL_TRACE", "RAINFALL_MISSING_CODE", "RAINFALL"]]
    .head(20)
)

Rainfall cleaning summary
-------------------------
Trace rainfall days: 574
Missing rainfall code days: 439
Missing rainfall after cleaning: 439
Unexpected negative rainfall values after cleaning: 0


,DATE,RAINFALL_RAW,RAINFALL_TRACE,RAINFALL_MISSING_CODE,RAINFALL
0,1961-01-01,-999.0,False,True,NaN
1,1961-01-02,-999.0,False,True,NaN
2,1961-01-03,-999.0,False,True,NaN
3,1961-01-04,-999.0,False,True,NaN
4,1961-01-05,0.0,False,False,0.0
5,1961-01-06,0.0,False,False,0.0
6,1961-01-07,0.0,False,False,0.0
7,1961-01-08,0.0,False,False,0.0
8,1961-01-09,0.0,False,False,0.0
9,1961-01-10,0.0,False,False,0.0


In [7]:
duplicate_dates = sci[sci["DATE"].duplicated(keep=False)].copy()

if len(duplicate_dates) > 0:
    display(
        duplicate_dates.sort_values("DATE")[
            ["DATE", "RAINFALL_RAW", "RAINFALL"]
        ].head(30)
    )
    raise ValueError("Duplicate DATE values found in Science Garden data. Please inspect before proceeding.")

# Reindex to complete daily calendar so missing dates are represented explicitly
full_dates = pd.date_range(
    start=sci["DATE"].min(),
    end=sci["DATE"].max(),
    freq="D"
)

sci = (
    sci
    .set_index("DATE")
    .reindex(full_dates)
    .rename_axis("DATE")
    .reset_index()
)

# Recreate date fields after reindexing
sci["YEAR"] = sci["DATE"].dt.year
sci["MONTH"] = sci["DATE"].dt.month
sci["DAY"] = sci["DATE"].dt.day

# Restore boolean flags after reindexing
for col in ["RAINFALL_TRACE", "RAINFALL_MISSING_CODE"]:
    sci[col] = sci[col].fillna(False).astype(bool)

# Wet-day flag
sci["WET_DAY"] = (sci["RAINFALL"] >= WET_DAY_THRESHOLD) & sci["RAINFALL"].notna()

print("After imposing continuous daily calendar")
print("----------------------------------------")
print(f"Rows: {len(sci):,}")
print(f"Date range: {sci['DATE'].min().date()} to {sci['DATE'].max().date()}")
print(f"Valid rainfall days: {sci['RAINFALL'].notna().sum():,}")
print(f"Wet days: {sci['WET_DAY'].sum():,}")

display(sci.head())

After imposing continuous daily calendar
----------------------------------------
Rows: 23,376
Date range: 1961-01-01 to 2024-12-31
Valid rainfall days: 22,937
Wet days: 8,487


,DATE,YEAR,MONTH,DAY,RAINFALL,TMAX,TMIN,RH,WIND_SPEED,WIND_DIRECTION,RAINFALL_RAW,RAINFALL_MISSING_CODE,RAINFALL_TRACE,WET_DAY
0,1961-01-01,1961,1,1,NaN,NaN,NaN,NaN,NaN,NaN,-999.0,True,False,False
1,1961-01-02,1961,1,2,NaN,NaN,NaN,NaN,NaN,NaN,-999.0,True,False,False
2,1961-01-03,1961,1,3,NaN,NaN,NaN,NaN,NaN,NaN,-999.0,True,False,False
3,1961-01-04,1961,1,4,NaN,NaN,NaN,NaN,NaN,NaN,-999.0,True,False,False
4,1961-01-05,1961,1,5,0.0,29.7,18.2,75.0,2.0,90.0,0.0,False,False,False


## Load IBTrACS tropical cyclone tracks

The IBTrACS western North Pacific file is used to flag days when a tropical cyclone centre was within selected radii of PAGASA Science Garden. The primary radius used in the analysis is 1000 km.

This is a tropical-cyclone proximity classification, not formal TC rainfall attribution.

In [8]:
tc = pd.read_csv(
    IBTRACS_PATH,
    skiprows=[1],
    usecols=["SID", "ISO_TIME", "LAT", "LON"],
    low_memory=False,
    na_values=["", " ", "NA", "NaN", "nan", "-999", -999]
)

tc.columns = tc.columns.str.strip().str.upper()

tc["ISO_TIME"] = pd.to_datetime(tc["ISO_TIME"], errors="coerce")
tc["LAT"] = pd.to_numeric(tc["LAT"], errors="coerce")
tc["LON"] = pd.to_numeric(tc["LON"], errors="coerce")

tc = tc.dropna(subset=["SID", "ISO_TIME", "LAT", "LON"]).copy()
tc["DATE"] = tc["ISO_TIME"].dt.floor("D")

# Limit to Science Garden data period with small buffer
buffer_days = 2
start_date = sci["DATE"].min() - pd.Timedelta(days=buffer_days)
end_date = sci["DATE"].max() + pd.Timedelta(days=buffer_days)

tc = tc[
    (tc["DATE"] >= start_date)
    & (tc["DATE"] <= end_date)
].copy()

print("Loaded cleaned IBTrACS track points")
print("-----------------------------------")
print(f"Rows: {len(tc):,}")
print(f"Date range: {tc['DATE'].min().date()} to {tc['DATE'].max().date()}")
print(f"Unique storms: {tc['SID'].nunique():,}")

display(tc.head())

Loaded cleaned IBTrACS track points
-----------------------------------
Rows: 155,218
Date range: 1961-01-13 to 2024-12-26
Unique storms: 2,275


,SID,ISO_TIME,LAT,LON,DATE
90111,1961014N07141,1961-01-13 12:00:00,7.0,141.0,1961-01-13
90112,1961014N07141,1961-01-13 15:00:00,7.2,140.2,1961-01-13
90113,1961014N07141,1961-01-13 18:00:00,7.3,139.4,1961-01-13
90114,1961014N07141,1961-01-13 21:00:00,7.4,138.5,1961-01-13
90115,1961014N07141,1961-01-14 00:00:00,7.6,137.0,1961-01-14


In [9]:
tc["dist_to_sci_garden_km"] = haversine_km(
    SCI_GARDEN_LAT,
    SCI_GARDEN_LON,
    tc["LAT"],
    tc["LON"]
)

print("Distance summary, km")
print("--------------------")
display(tc["dist_to_sci_garden_km"].describe())

print("Closest IBTrACS track points to Science Garden")
display(
    tc.sort_values("dist_to_sci_garden_km")
    [["SID", "ISO_TIME", "DATE", "LAT", "LON", "dist_to_sci_garden_km"]]
    .head(20)
)

Distance summary, km
--------------------


count    155218.000000
mean       2422.196212
std        1734.455270
min           7.735485
25%        1111.462263
50%        1971.876524
75%        3359.052893
max       15319.569522
Name: dist_to_sci_garden_km, dtype: float64

Closest IBTrACS track points to Science Garden


,SID,ISO_TIME,DATE,LAT,LON,dist_to_sci_garden_km
212651,2008169N08135,2008-06-22 00:00:00,2008-06-22,14.7,121.0,7.735485
184137,1995264N06174,1995-09-30 18:00:00,1995-09-30,14.7,121.1,7.735485
121589,1972174N11137,1972-06-25 03:00:00,1972-06-25,14.6,121.1,7.736338
241510,2022299N11134,2022-10-29 12:00:00,2022-10-29,14.6,121.1,7.736338
99910,1964269N12142,1964-09-29 03:00:00,1964-09-29,14.6,121.1,7.736338
197362,2000305N06136,2000-11-02 21:00:00,2000-11-02,14.7,120.9,17.066146
241682,2023101N14127,2023-04-13 06:00:00,2023-04-13,14.6,121.2,17.069626
152980,1985183N08136,1985-07-05 12:00:00,1985-07-05,14.6,121.2,17.069626
98416,1964177N09142,1964-06-29 18:00:00,1964-06-29,14.6,120.9,17.069626
100764,1964332N12120,1964-11-28 12:00:00,1964-11-28,14.6,120.9,17.069626


In [10]:
# First collapse each storm to its closest position per day
tc_daily_storm = (
    tc.sort_values("dist_to_sci_garden_km")
    .groupby(["DATE", "SID"], as_index=False)
    .first()
)

summary_rows = []

for date, g in tc_daily_storm.groupby("DATE", sort=True):
    g = g.copy()
    nearest = g.loc[g["dist_to_sci_garden_km"].idxmin()]
    
    row = {
        "DATE": date,
        "tc_min_dist_km": nearest["dist_to_sci_garden_km"],
        "tc_sid_nearest": nearest["SID"],
        "tc_lat_nearest": nearest["LAT"],
        "tc_lon_nearest": nearest["LON"],
    }
    
    for radius in TC_RADII_KM:
        within_radius = g[g["dist_to_sci_garden_km"] <= radius].copy()
        
        row[f"tc_{radius}km"] = len(within_radius) > 0
        row[f"tc_count_{radius}km"] = within_radius["SID"].nunique()
        
        if len(within_radius) > 0:
            sid_list = (
                within_radius["SID"]
                .dropna()
                .astype(str)
                .sort_values()
                .unique()
                .tolist()
            )
            row[f"tc_sids_{radius}km"] = ", ".join(sid_list)
        else:
            row[f"tc_sids_{radius}km"] = pd.NA
    
    summary_rows.append(row)

tc_daily = pd.DataFrame(summary_rows)

print("Created daily TC proximity summary")
print("----------------------------------")
print(f"Dates with any WNP TC record: {len(tc_daily):,}")

for radius in TC_RADII_KM:
    print(f"Dates with TC within {radius:>4} km: {tc_daily[f'tc_{radius}km'].sum():,}")

display(tc_daily.head())

Created daily TC proximity summary
----------------------------------
Dates with any WNP TC record: 12,024
Dates with TC within  250 km: 464
Dates with TC within  500 km: 1,598
Dates with TC within 1000 km: 4,701


,DATE,tc_min_dist_km,tc_sid_nearest,tc_lat_nearest,tc_lon_nearest,tc_250km,tc_count_250km,tc_sids_250km,tc_500km,tc_count_500km,tc_sids_500km,tc_1000km,tc_count_1000km,tc_sids_1000km
0,1961-01-13,2066.575543,1961014N07141,7.4,138.5,False,0,<NA>,False,0,<NA>,False,0,<NA>
1,1961-01-14,1424.190960,1961014N07141,9.0,132.8,False,0,<NA>,False,0,<NA>,False,0,<NA>
2,1961-01-15,1276.736042,1961014N07141,10.0,131.8,False,0,<NA>,False,0,<NA>,False,0,<NA>
3,1961-01-16,1250.021237,1961014N07141,10.6,131.8,False,0,<NA>,False,0,<NA>,False,0,<NA>
4,1961-01-17,1334.459172,1961014N07141,11.7,133.0,False,0,<NA>,False,0,<NA>,False,0,<NA>


## Merge rainfall and TC proximity flags

The prepared dataset contains one row per calendar day. TC proximity flags are filled as `False` when there is no TC within the specified radius or no IBTrACS western North Pacific TC record on that date.

In [11]:
prepared = sci.merge(tc_daily, on="DATE", how="left")

for radius in TC_RADII_KM:
    flag_col = f"tc_{radius}km"
    count_col = f"tc_count_{radius}km"
    sid_col = f"tc_sids_{radius}km"
    
    prepared[flag_col] = prepared[flag_col].fillna(False).astype(bool)
    prepared[count_col] = prepared[count_col].fillna(0).astype(int)
    
    if sid_col in prepared.columns:
        prepared[sid_col] = prepared[sid_col].astype("string")

string_columns = ["tc_sid_nearest"] + [f"tc_sids_{radius}km" for radius in TC_RADII_KM]

for col in string_columns:
    if col in prepared.columns:
        prepared[col] = prepared[col].astype("string")

prepared["tc_min_dist_km"] = pd.to_numeric(prepared["tc_min_dist_km"], errors="coerce")

prepared = prepared.sort_values("DATE").reset_index(drop=True)

print("Prepared daily dataset")
print("----------------------")
print(f"Rows: {len(prepared):,}")
print(f"Date range: {prepared['DATE'].min().date()} to {prepared['DATE'].max().date()}")
print(f"Valid rainfall days: {prepared['RAINFALL'].notna().sum():,}")
print(f"Wet days: {prepared['WET_DAY'].sum():,}")

for radius in TC_RADII_KM:
    print(f"Days with TC within {radius:>4} km: {prepared[f'tc_{radius}km'].sum():,}")

display(
    prepared[
        [
            "DATE",
            "RAINFALL",
            "RAINFALL_RAW",
            "RAINFALL_TRACE",
            "WET_DAY",
            "tc_min_dist_km",
            "tc_sid_nearest",
            "tc_250km",
            "tc_500km",
            "tc_1000km",
            "tc_count_1000km",
            "tc_sids_1000km",
        ]
    ].head(20)
)

Prepared daily dataset
----------------------
Rows: 23,376
Date range: 1961-01-01 to 2024-12-31
Valid rainfall days: 22,937
Wet days: 8,487
Days with TC within  250 km: 464
Days with TC within  500 km: 1,598
Days with TC within 1000 km: 4,701


/tmp/ipykernel_1387678/2602831212.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prepared[flag_col] = prepared[flag_col].fillna(False).astype(bool)
/tmp/ipykernel_1387678/2602831212.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prepared[flag_col] = prepared[flag_col].fillna(False).astype(bool)
/tmp/ipykernel_1387678/2602831212.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.

,DATE,RAINFALL,RAINFALL_RAW,RAINFALL_TRACE,WET_DAY,tc_min_dist_km,tc_sid_nearest,tc_250km,tc_500km,tc_1000km,tc_count_1000km,tc_sids_1000km
0,1961-01-01,NaN,-999.0,False,False,NaN,<NA>,False,False,False,0,<NA>
1,1961-01-02,NaN,-999.0,False,False,NaN,<NA>,False,False,False,0,<NA>
2,1961-01-03,NaN,-999.0,False,False,NaN,<NA>,False,False,False,0,<NA>
3,1961-01-04,NaN,-999.0,False,False,NaN,<NA>,False,False,False,0,<NA>
4,1961-01-05,0.0,0.0,False,False,NaN,<NA>,False,False,False,0,<NA>
5,1961-01-06,0.0,0.0,False,False,NaN,<NA>,False,False,False,0,<NA>
6,1961-01-07,0.0,0.0,False,False,NaN,<NA>,False,False,False,0,<NA>
7,1961-01-08,0.0,0.0,False,False,NaN,<NA>,False,False,False,0,<NA>
8,1961-01-09,0.0,0.0,False,False,NaN,<NA>,False,False,False,0,<NA>
9,1961-01-10,0.0,0.0,False,False,NaN,<NA>,False,False,False,0,<NA>


In [12]:
validation_checks = {
    "one_row_per_day": prepared["DATE"].is_unique,
    "date_is_monotonic": prepared["DATE"].is_monotonic_increasing,
    "no_negative_cleaned_rainfall": not (
        prepared["RAINFALL"].notna() & (prepared["RAINFALL"] < 0)
    ).any(),
    "trace_values_cleaned_to_zero": (
        prepared.loc[prepared["RAINFALL_TRACE"], "RAINFALL"].eq(0.0).all()
        if prepared["RAINFALL_TRACE"].any()
        else True
    ),
    "tc_1000km_boolean": prepared["tc_1000km"].dtype == bool,
}

print("Validation checks")
print("-----------------")
for check, passed in validation_checks.items():
    print(f"{check}: {passed}")

if not all(validation_checks.values()):
    raise ValueError("One or more validation checks failed. Please inspect the prepared dataset.")## Data completeness summaries

The following completeness summaries are exported as derived tables. These do not contain daily rainfall values.

Validation checks
-----------------
one_row_per_day: True
date_is_monotonic: True
no_negative_cleaned_rainfall: True
trace_values_cleaned_to_zero: True
tc_1000km_boolean: True


## Data completeness summaries

The following completeness summaries are exported as derived tables. These do not contain daily rainfall values.

In [13]:
annual_completeness = (
    prepared
    .groupby("YEAR")
    .agg(
        valid_rainfall_days=("RAINFALL", "count"),
        trace_days=("RAINFALL_TRACE", "sum"),
        wet_days=("WET_DAY", "sum"),
        tc1000_days=("tc_1000km", "sum"),
    )
    .reset_index()
)

annual_completeness["expected_days"] = annual_completeness["YEAR"].apply(expected_days_in_year)
annual_completeness["missing_rainfall_days"] = (
    annual_completeness["expected_days"] - annual_completeness["valid_rainfall_days"]
)
annual_completeness["valid_fraction"] = (
    annual_completeness["valid_rainfall_days"] / annual_completeness["expected_days"]
)
annual_completeness["is_valid_year"] = (
    annual_completeness["valid_fraction"] >= MIN_VALID_FRACTION
)

monthly_completeness = (
    prepared
    .groupby(["YEAR", "MONTH"])
    .agg(
        valid_rainfall_days=("RAINFALL", "count"),
        trace_days=("RAINFALL_TRACE", "sum"),
        wet_days=("WET_DAY", "sum"),
        tc1000_days=("tc_1000km", "sum"),
    )
    .reset_index()
)

monthly_completeness["expected_days"] = monthly_completeness.apply(
    lambda row: expected_days_in_month(row["YEAR"], row["MONTH"]),
    axis=1
)
monthly_completeness["missing_rainfall_days"] = (
    monthly_completeness["expected_days"] - monthly_completeness["valid_rainfall_days"]
)
monthly_completeness["valid_fraction"] = (
    monthly_completeness["valid_rainfall_days"] / monthly_completeness["expected_days"]
)
monthly_completeness["is_valid_month"] = (
    monthly_completeness["valid_fraction"] >= MIN_VALID_FRACTION
)

display(annual_completeness.head())
display(monthly_completeness.head())

annual_completeness.to_csv(ANNUAL_COMPLETENESS_PATH, index=False)
monthly_completeness.to_csv(MONTHLY_COMPLETENESS_PATH, index=False)

print("Saved completeness summaries:")
print(f"- {ANNUAL_COMPLETENESS_PATH}")
print(f"- {MONTHLY_COMPLETENESS_PATH}")

,YEAR,valid_rainfall_days,trace_days,wet_days,tc1000_days,expected_days,missing_rainfall_days,valid_fraction,is_valid_year
0,1961,361,0,138,101,365,4,0.989041,True
1,1962,365,0,121,85,365,0,1.000000,True
2,1963,365,0,140,67,365,0,1.000000,True
3,1964,366,0,143,110,366,0,1.000000,True
4,1965,365,0,127,93,365,0,1.000000,True


,YEAR,MONTH,valid_rainfall_days,trace_days,wet_days,tc1000_days,expected_days,missing_rainfall_days,valid_fraction,is_valid_month
0,1961,1,27,0,0,0,31,4,0.870968,False
1,1961,2,28,0,3,0,28,0,1.000000,True
2,1961,3,31,0,7,0,31,0,1.000000,True
3,1961,4,30,0,2,0,30,0,1.000000,True
4,1961,5,31,0,12,11,31,0,1.000000,True


Saved completeness summaries:
- /home/jupyter-bbr/source/science-garden-rainfall-trends/output/tables/annual_data_completeness.csv
- /home/jupyter-bbr/source/science-garden-rainfall-trends/output/tables/monthly_data_completeness.csv


In [14]:
prepared.to_csv(PREPARED_CSV_PATH, index=False)
prepared.to_pickle(PREPARED_PKL_PATH)

metadata = {
    "science_garden_input": str(SCI_GARDEN_PATH),
    "ibtracs_input": str(IBTRACS_PATH),
    "prepared_csv": str(PREPARED_CSV_PATH),
    "prepared_pickle": str(PREPARED_PKL_PATH),
    "date_start": str(prepared["DATE"].min().date()),
    "date_end": str(prepared["DATE"].max().date()),
    "n_days": int(len(prepared)),
    "valid_rainfall_days": int(prepared["RAINFALL"].notna().sum()),
    "trace_rainfall_days": int(prepared["RAINFALL_TRACE"].sum()),
    "wet_day_threshold_mm": WET_DAY_THRESHOLD,
    "tc_radii_km": TC_RADII_KM,
    "primary_tc_radius_km": PRIMARY_TC_RADIUS_KM,
    "station_lat": SCI_GARDEN_LAT,
    "station_lon": SCI_GARDEN_LON,
    "note": (
        "Prepared daily dataset contains PAGASA-derived daily rainfall values "
        "and should not be redistributed unless data-sharing permissions allow."
    ),
}

with open(PREPARED_METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved prepared dataset locally:")
print(f"- {PREPARED_CSV_PATH}")
print(f"- {PREPARED_PKL_PATH}")
print(f"- {PREPARED_METADATA_PATH}")

Saved prepared dataset locally:
- /home/jupyter-bbr/source/science-garden-rainfall-trends/output/intermediate/science_garden_daily_prepared.csv
- /home/jupyter-bbr/source/science-garden-rainfall-trends/output/intermediate/science_garden_daily_prepared.pkl
- /home/jupyter-bbr/source/science-garden-rainfall-trends/output/intermediate/science_garden_daily_prepared_metadata.json


## Output note

The prepared daily dataset has been saved to `output/intermediate/`. This folder should remain local and should not be committed to GitHub because it contains PAGASA-derived daily rainfall values.

The next notebook, `02_analysis_and_figures.ipynb`, will load the prepared dataset and generate climatology, rainfall indices, trend tables, and manuscript figures.